### Großstadtbauern – Lösung Schritt für Schritt

*44. Bundeswettbewerb Informatik (2025), 1. Runde, Aufgabe 5*

Im Jahr 2238 bauen alle Menschen ihre Nahrungsmittel in Kleinanlagen selbst an. Der Anbau ist von den Jahreszeiten
vollkommen unabhängig. Nemo ist in seinem Wohnblock für die Anbauplanung zuständig.

Der Wohnblock besitzt drei Gewächsanlagen. Jede Anlage kann für beliebige Gemüse- oder Obstsorten verwendet werden, aber
innerhalb eines dreimonatigen Anbauzyklus kann pro Anlage nur eine Gemüse- oder Obstsorte angebaut werden. Nemo steuert den
Anbau immer so, dass Anbauzyklen am Monatsanfang beginnen, keine zwei Anbauzyklen gleichzeitig beginnen und jedes Jahr
derselbe Plan verwendet wird, zum Beispiel so:

| | Jan | Feb | Mär | Apr | Mai | Jun | Jul | Aug | Sep | Okt | Nov | Dez |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Anlage 1 | Äpfel | Äpfel | Äpfel | Banane | Banane | Banane | Kirschen | Kirschen | Kirschen | Kaki | Kaki | Kaki |
| Anlage 2 | Kartoffeln | Kartoffeln | Kohlrabi | Kohlrabi | Kohlrabi | Fenchel | Fenchel | Fenchel | Rote Beete | Rote Beete | Rote Beete | Kartoffeln |
| Anlage 3 | Tomaten | Paprika | Paprika | Paprika | Zwiebeln | Zwiebeln | Zwiebeln | Zucchini | Zucchini | Zucchini | Tomaten | Tomaten |

(Anlage 1 beginnt ihre Zyklen im Januar, April, Juli, Oktober; Anlage 2 im März, Juni, September, Dezember;
Anlage 3 im Februar, Mai, August, November. Die Zyklen von Anlage 2 und 3 gehen über den Jahreswechsel.)

Nemo hat eine Wunschliste von Gerichten aus seinem Wohnblock erhalten. Ein Gericht besteht aus genau 3 Zutaten. Nemo möchte
den Anbau so optimieren, dass möglichst viele Gerichte von der Wunschliste in mindestens einem Monat im Jahr gekocht werden
können. Zutaten müssen in dem Monat verbraucht werden, in dem sie angebaut wurden.

#### Aufgabe 5

Schreibe ein Programm, das eine Wunschliste einliest und einen bestmöglichen Anbauplan berechnet.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `bauern1.txt` bis `bauern6.txt` (`bauern5.txt` und `bauern6.txt` sind laut BWINF-Webseite
optionale Herausforderungen). Die erste Zeile enthält die Anzahl `n` der Gemüsesorten, dann folgt je Zeile eine Sorte. Danach
kommt die Anzahl `m` der Gerichte und je Zeile ein Gericht aus drei Gemüsesorten, getrennt durch Leerzeichen:

```
4
Kartoffel
Zwiebel
Möhre
Reis
4
Kartoffel Zwiebel Möhre
Kartoffel Zwiebel Reis
Kartoffel Möhre Reis
Zwiebel Möhre Reis
```

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Gute Modellierung:** Jeden Monat beginnt genau ein Anbauzyklus (12 Monate, 3 Anlagen mit je 4 Zyklen). Nummeriert man die
  Zyklen nach ihrem Startmonat, ist ein Anbauplan einfach eine Liste aus **12 Pflanzen**. In jedem Monat wachsen die Pflanzen von
  drei Zyklen, die direkt nacheinander begonnen haben. Die Liste ist dabei ein **Kreis**: Auf Dezember folgt wieder Januar.
- **Backtracking mit Rekursion:** Monat für Monat entscheiden, ob dort ein neues Gericht gekocht werden soll und welches. Eine
  Funktion ruft sich dafür selbst für den nächsten Monat auf.
- **Branch and Bound (Suche mit Schranke):** Man merkt sich den besten bisher gefundenen Plan. Kann eine angefangene Suche
  höchstens noch genauso viele Gerichte schaffen, bricht man sie ab.
- **Symmetrien ausnutzen:** Verschiebt man einen Plan um einen Monat, kann man dieselben Gerichte kochen. Auch rückwärts gelesen
  bleibt der Plan gleich gut. Solche gleichwertigen Pläne muss man nur einmal untersuchen.

**Nützliche Funktionen und Bibliotheken**
- Listen, Tupel, `set` (Mengen vergleichen: `set(gericht) == pflanzen`), Rest beim Teilen `%` für den Kreis
- `itertools.permutations(gericht)` liefert die 6 Reihenfolgen der drei Zutaten eines Gerichts

## Die Idee

1. Den Plan als Kreis aus 12 Pflanzen darstellen und zum Prüfen als Tabelle wie auf dem Aufgabenblatt ausgeben.
2. Alle Pläne durchzuprobieren dauert viel zu lange (bei 50 Sorten 50¹² Pläne). Darum gehen wir umgekehrt vor und probieren
   **Gerichte** statt Pflanzen: Für jeden Monat wählen wir ein noch nicht gekochtes Gericht und eine Reihenfolge seiner Zutaten,
   die zu den schon festgelegten Pflanzen passt, oder wir kochen in diesem Monat nichts Neues.
3. Mit einer **Schranke** brechen wir Suchen ab, die den besten bisherigen Plan nicht mehr übertreffen können.
4. **Symmetrien** sparen weitere Arbeit: Das erste Gericht kommt immer in den Januar.

So ähnlich gehen auch die offiziellen Lösungshinweise vor.

## Schritt 1: Die Eingabe lesen

Wir lesen die Gemüsesorten in eine Liste und die Gerichte als Tupel aus drei Texten.

In [ ]:
def lies_eingabe(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = []
        for zeile in f.read().splitlines():
            if zeile.split():                       # leere Zeilen überspringen
                zeilen.append(zeile.split())
    n = int(zeilen[0][0])
    gemuese = []
    for i in range(1, n + 1):
        gemuese.append(zeilen[i][0])
    m = int(zeilen[n + 1][0])
    gerichte = []
    for i in range(n + 2, n + 2 + m):
        gerichte.append(tuple(zeilen[i]))
    return gemuese, gerichte


gemuese, gerichte = lies_eingabe('beispieldaten/bauern1.txt')
print(gemuese)
for g in gerichte:
    print(g)

['Kartoffel', 'Zwiebel', 'Möhre', 'Reis']
('Kartoffel', 'Zwiebel', 'Möhre')
('Kartoffel', 'Zwiebel', 'Reis')
('Kartoffel', 'Möhre', 'Reis')
('Zwiebel', 'Möhre', 'Reis')


## Schritt 2: Der Anbauplan als Kreis

Weil keine zwei Zyklen gleichzeitig beginnen und es 12 Zyklen gibt (3 Anlagen mal 4), beginnt in **jedem Monat genau ein**
Zyklus. Wir nummerieren die Zyklen nach ihrem Startmonat: Zyklus 0 beginnt im Januar, Zyklus 1 im Februar, …, Zyklus 11 im
Dezember. Diese Nummern nennen wir **Slots**. Ein Plan ist dann eine Liste `plan` mit 12 Pflanzen, `plan[s]` wächst im Slot `s`.

Im Monat `monat` (0 = Januar) wachsen die Pflanzen der Slots `monat − 2`, `monat − 1` und `monat`, denn jeder Zyklus dauert drei
Monate. Im Januar sind das die Slots 10 (November), 11 (Dezember) und 0 (Januar). Mit `% 12` rechnen wir im Kreis.

Jede Anlage hat jeden dritten Slot: Anlage 1 die Slots 0, 3, 6, 9 (Januar, April, Juli, Oktober), Anlage 2 die Slots 2, 5, 8, 11
(März, Juni, September, Dezember), Anlage 3 die Slots 1, 4, 7, 10. `slot_von(anlage, monat)` geht vom Monat so weit zurück, bis
ein Slot der Anlage beginnt.

Zur Kontrolle schreiben wir den Plan vom Aufgabenblatt als Liste und geben ihn als Tabelle aus (Monate untereinander, damit die
Spalten nicht zu breit werden).

In [ ]:
MONATE = ['Jan', 'Feb', 'Mär', 'Apr', 'Mai', 'Jun', 'Jul', 'Aug', 'Sep', 'Okt', 'Nov', 'Dez']
ERSTER_SLOT = [0, 2, 1]             # Anlage 1 beginnt im Januar, Anlage 2 im März, Anlage 3 im Februar


def slots(monat):
    """Die drei Slots, deren Pflanzen im Monat wachsen."""
    return [(monat - 2) % 12, (monat - 1) % 12, monat]


def slot_von(anlage, monat):
    """Slot, dessen Pflanze im Monat in der Anlage (1 bis 3) wächst."""
    zurueck = (monat - ERSTER_SLOT[anlage - 1]) % 3
    return (monat - zurueck) % 12


def zeige_plan(plan):
    breite = len('beliebig') + 2
    for pflanze in plan:
        if pflanze is not None and len(pflanze) + 2 > breite:
            breite = len(pflanze) + 2
    print('Monat'.ljust(6) + 'Anlage 1'.ljust(breite) + 'Anlage 2'.ljust(breite) + 'Anlage 3')
    for monat in range(12):
        zeile = MONATE[monat].ljust(6)
        for anlage in [1, 2, 3]:
            pflanze = plan[slot_von(anlage, monat)]
            if pflanze is None:
                pflanze = 'beliebig'
            zeile = zeile + pflanze.ljust(breite)
        print(zeile)


beispiel = ['Äpfel', 'Paprika', 'Kohlrabi', 'Banane', 'Zwiebeln', 'Fenchel',
            'Kirschen', 'Zucchini', 'Rote Beete', 'Kaki', 'Tomaten', 'Kartoffeln']
zeige_plan(beispiel)

Monat Anlage 1    Anlage 2    Anlage 3
Jan   Äpfel       Kartoffeln  Tomaten     
Feb   Äpfel       Kartoffeln  Paprika     
Mär   Äpfel       Kohlrabi    Paprika     
Apr   Banane      Kohlrabi    Paprika     
Mai   Banane      Kohlrabi    Zwiebeln    
Jun   Banane      Fenchel     Zwiebeln    
Jul   Kirschen    Fenchel     Zwiebeln    
Aug   Kirschen    Fenchel     Zucchini    
Sep   Kirschen    Rote Beete  Zucchini    
Okt   Kaki        Rote Beete  Zucchini    
Nov   Kaki        Rote Beete  Tomaten     
Dez   Kaki        Kartoffeln  Tomaten     


Das ist genau der Plan vom Aufgabenblatt, nur gekippt. Mit 12 Pflanzen in einem Kreis ist die Überlappung der Zyklen, auch über
den Jahreswechsel hinweg, also vollständig beschrieben. Wenn eine Pflanze bei der Suche noch nicht festgelegt ist, steht in der
Liste `None`, und die Tabelle zeigt „beliebig“.

## Schritt 3: Welche Gerichte sind kochbar?

Ein Gericht ist in einem Monat kochbar, wenn seine drei Zutaten genau die drei Pflanzen sind, die in diesem Monat wachsen. Mit
Mengen ist der Vergleich einfach, weil die Reihenfolge dann keine Rolle spielt: `set(gericht) == pflanzen`.

Für `bauern1.txt` probieren wir den Kreis aus Kartoffel, Zwiebel, Möhre, Reis, dreimal hintereinander. Je drei Nachbarn im Kreis
sind immer drei verschiedene der vier Sorten, und das sind genau die vier Gerichte.

In [ ]:
def gericht_im_monat(plan, monat, gerichte):
    """Das Gericht, das im Monat kochbar ist, oder None."""
    pflanzen = set()
    for s in slots(monat):
        pflanzen.add(plan[s])
    for g in gerichte:
        if set(g) == pflanzen:
            return g
    return None


def kochbare(plan, gerichte):
    """Liste der Gerichte, die in mindestens einem Monat kochbar sind (jedes nur einmal)."""
    ergebnis = []
    for monat in range(12):
        g = gericht_im_monat(plan, monat, gerichte)
        if g is not None and g not in ergebnis:
            ergebnis.append(g)
    return ergebnis


kreis = ['Kartoffel', 'Zwiebel', 'Möhre', 'Reis'] * 3
for monat in range(12):
    print(MONATE[monat], gericht_im_monat(kreis, monat, gerichte))
print('kochbare Gerichte:', len(kochbare(kreis, gerichte)))

Jan ('Kartoffel', 'Möhre', 'Reis')
Feb ('Kartoffel', 'Zwiebel', 'Reis')
Mär ('Kartoffel', 'Zwiebel', 'Möhre')
Apr ('Zwiebel', 'Möhre', 'Reis')
Mai ('Kartoffel', 'Möhre', 'Reis')
Jun ('Kartoffel', 'Zwiebel', 'Reis')
Jul ('Kartoffel', 'Zwiebel', 'Möhre')
Aug ('Zwiebel', 'Möhre', 'Reis')
Sep ('Kartoffel', 'Möhre', 'Reis')
Okt ('Kartoffel', 'Zwiebel', 'Reis')
Nov ('Kartoffel', 'Zwiebel', 'Möhre')
Dez ('Zwiebel', 'Möhre', 'Reis')
kochbare Gerichte: 4


## Schritt 4: Gerichte statt Pflanzen ausprobieren

Alle Pläne durchzuprobieren ist hoffnungslos: Bei 50 Sorten (`bauern4.txt`) gibt es 50¹² Pläne, eine Zahl mit 21 Stellen.

Wir drehen die Sache um. Wir gehen die Monate der Reihe nach durch und entscheiden für jeden Monat:

- **Ein neues Gericht kochen:** Wir wählen ein Gericht, das noch nicht gekocht wird, und eine der 6 Reihenfolgen seiner Zutaten
  für die drei Slots des Monats. Die Reihenfolge muss zu den Slots passen, die frühere Monate schon festgelegt haben.
- **Nichts Neues kochen:** Die Slots bleiben, wie sie sind (vielleicht noch `None`).

Am Ende hat jeder gewählte Monat ein anderes Gericht, die Zahl der Gerichte ist also die Zahl dieser Monate. Freie Slots dürfen
beliebig belegt werden. Der beste Plan ist auf diesem Weg sicher dabei: Für jedes Gericht des besten Plans wählt man einen Monat,
in dem es kochbar ist.

Dazu brauchen wir zwei kleine Hilfsfunktionen: `passt` prüft, ob eine Reihenfolge zu den schon belegten Slots passt, und `setze`
trägt drei Werte in die Slots ein (auch zum Zurücksetzen).

In [ ]:
from itertools import permutations


def passt(plan, s, reihenfolge):
    for j in range(3):
        if plan[s[j]] is not None and plan[s[j]] != reihenfolge[j]:
            return False
    return True


def setze(plan, s, werte):
    for j in range(3):
        plan[s[j]] = werte[j]


for r in permutations(('Kartoffel', 'Zwiebel', 'Möhre')):
    print(r)

('Kartoffel', 'Zwiebel', 'Möhre')
('Kartoffel', 'Möhre', 'Zwiebel')
('Zwiebel', 'Kartoffel', 'Möhre')
('Zwiebel', 'Möhre', 'Kartoffel')
('Möhre', 'Kartoffel', 'Zwiebel')
('Möhre', 'Zwiebel', 'Kartoffel')


## Schritt 5: Die Suche mit Schranke und Verschiebung

`suche(monat, plan, offen, anzahl, beste)` bekommt den Monat, um den es gerade geht, den angefangenen Plan, die Liste `offen` der
Gerichte, die noch gewählt werden dürfen, und die Zahl `anzahl` der schon gewählten Gerichte. In `beste` steht der beste bisher
gefundene Plan: `beste[0]` ist seine Anzahl, `beste[1]` der Plan.

- **Schranke:** In den Monaten `monat` bis 11 kommt höchstens ein neues Gericht pro Monat dazu, und nicht mehr, als in `offen`
  stehen. Wird `anzahl` plus diese Obergrenze nicht größer als `beste[0]`, kann diese Suche den besten Plan nicht übertreffen.
- **Verschiebung:** Verschiebt man jede Pflanze in den nächsten Slot, sind alle Gerichte einen Monat später kochbar, aber es sind
  dieselben. Darum darf man verlangen, dass im **Januar** ein Gericht gekocht wird. Und mehr noch: Hat man alle Pläne mit dem
  ersten Gericht im Januar untersucht, kennt man (bis auf Verschiebung) alle Pläne, die dieses Gericht enthalten. Wählt man dann im
  Januar das zweite Gericht, braucht das erste gar nicht mehr vorzukommen. Deshalb bleiben bei der Wahl im Januar nur die Gerichte
  **hinter** dem gewählten in `offen`.

In `aufrufe` zählen wir, wie oft `suche` aufgerufen wird.

In [ ]:
aufrufe = [0]


def suche(monat, plan, offen, anzahl, beste):
    aufrufe[0] = aufrufe[0] + 1
    if anzahl + min(12 - monat, len(offen)) <= beste[0]:
        return                                      # kann nicht besser werden als der beste Plan
    if monat == 12:                                 # alle Monate entschieden: neuer bester Plan
        beste[0] = anzahl
        beste[1] = plan[:]
        return
    s = slots(monat)
    for i in range(len(offen)):                     # ein neues Gericht kochen
        for reihenfolge in permutations(offen[i]):
            if passt(plan, s, reihenfolge):
                alt = [plan[s[0]], plan[s[1]], plan[s[2]]]
                setze(plan, s, reihenfolge)
                if monat == 0:
                    rest = offen[i + 1:]            # Verschiebung: nur die Gerichte dahinter
                else:
                    rest = offen[:i] + offen[i + 1:]
                suche(monat + 1, plan, rest, anzahl + 1, beste)
                setze(plan, s, alt)                 # zurücknehmen (Backtracking)
    if monat > 0:                                   # nichts Neues kochen (im Januar nicht nötig)
        suche(monat + 1, plan, offen, anzahl, beste)


for nr in ['1', '2', '3']:
    gemuese, gerichte = lies_eingabe('beispieldaten/bauern' + nr + '.txt')
    aufrufe[0] = 0
    beste = [0, [None] * 12]
    suche(0, [None] * 12, gerichte, 0, beste)
    print('bauern' + nr + '.txt:', beste[0], 'von', len(gerichte), 'Gerichten,', aufrufe[0], 'Aufrufe')

bauern1.txt: 4 von 4 Gerichten, 39 Aufrufe
bauern2.txt: 6 von 6 Gerichten, 614 Aufrufe
bauern3.txt: 8 von 19 Gerichten, 328812 Aufrufe


Die Ergebnisse stimmen schon (4, 6 und 8 Gerichte, siehe die Tabelle am Ende), aber für `bauern3.txt` sind es schon über 300 000
Aufrufe. Für die größeren Beispiele brauchen wir mehr Abkürzungen.

## Schritt 6: Schneller durch Spiegelung und eine bessere Schranke

**Spiegelung:** Liest man den Kreis rückwärts, wachsen in jedem Monat wieder drei benachbarte Pflanzen, also können dieselben
Gerichte gekocht werden. Die Januar-Slots 10, 11, 0 werden dabei zu 0, 11, 10: Die Reihenfolge `(a, b, c)` im Januar und die
umgekehrte `(c, b, a)` führen zu gleich guten Plänen. Wir nehmen nur die, bei der die erste Zutat im Alphabet vor der letzten
steht. Das halbiert die Arbeit.

**Bessere Schranke:** Bisher haben wir angenommen, dass in jedem übrigen Monat noch ein neues Gericht dazukommen kann. Oft geht das
aber nicht mehr, weil die schon belegten Slots des Monats zu keinem offenen Gericht passen. `noch_moeglich(plan, monat, offen)`
prüft, ob es ein offenes Gericht gibt, das alle schon belegten Pflanzen des Monats enthält. Nur solche Monate zählen für die
Obergrenze.

In [ ]:
def noch_moeglich(plan, monat, offen):
    belegt = []
    for s in slots(monat):
        if plan[s] is not None:
            belegt.append(plan[s])
    for g in offen:
        alle_drin = True
        for pflanze in belegt:
            if pflanze not in g:
                alle_drin = False
        if alle_drin:
            return True
    return False


def suche2(monat, plan, offen, anzahl, beste):
    aufrufe[0] = aufrufe[0] + 1
    grenze = 0                                      # Monate, in denen noch ein neues Gericht möglich ist
    for m in range(monat, 12):
        if noch_moeglich(plan, m, offen):
            grenze = grenze + 1
    if anzahl + min(grenze, len(offen)) <= beste[0]:
        return
    if monat == 12:
        beste[0] = anzahl
        beste[1] = plan[:]
        return
    s = slots(monat)
    for i in range(len(offen)):
        for reihenfolge in permutations(offen[i]):
            if monat == 0 and reihenfolge[0] > reihenfolge[2]:
                continue                            # Spiegelung: die umgekehrte Reihenfolge genügt
            if passt(plan, s, reihenfolge):
                alt = [plan[s[0]], plan[s[1]], plan[s[2]]]
                setze(plan, s, reihenfolge)
                if monat == 0:
                    rest = offen[i + 1:]
                else:
                    rest = offen[:i] + offen[i + 1:]
                suche2(monat + 1, plan, rest, anzahl + 1, beste)
                setze(plan, s, alt)
    if monat > 0:
        suche2(monat + 1, plan, offen, anzahl, beste)


for nr in ['1', '2', '3']:
    gemuese, gerichte = lies_eingabe('beispieldaten/bauern' + nr + '.txt')
    aufrufe[0] = 0
    beste = [0, [None] * 12]
    suche2(0, [None] * 12, gerichte, 0, beste)
    print('bauern' + nr + '.txt:', beste[0], 'von', len(gerichte), 'Gerichten,', aufrufe[0], 'Aufrufe')

bauern1.txt: 4 von 4 Gerichten, 27 Aufrufe
bauern2.txt: 6 von 6 Gerichten, 362 Aufrufe
bauern3.txt: 8 von 19 Gerichten, 17928 Aufrufe


Bei `bauern3.txt` sind es jetzt knapp 18 000 statt fast 330 000 Aufrufe. Jeder Aufruf ist zwar etwas teurer (wegen
`noch_moeglich`), insgesamt geht es aber viel schneller.

## Schritt 7: Alle Beispiele

Für jedes Beispiel geben wir die Zahl der kochbaren Gerichte, den Plan als Tabelle und die Gerichte mit ihren Monaten aus.
Gezählt wird mit `kochbare` aus Schritt 3 am fertigen Plan, also unabhängig von der Suche. `bauern4.txt` und `bauern5.txt`
dauern mit einem auf dem Rechner installierten Python einige Sekunden, im Browser länger.

In [ ]:
def bearbeite(dateiname):
    gemuese, gerichte = lies_eingabe(dateiname)
    aufrufe[0] = 0
    beste = [0, [None] * 12]
    suche2(0, [None] * 12, gerichte, 0, beste)
    plan = beste[1]
    gekocht = kochbare(plan, gerichte)
    print(dateiname + ':', len(gemuese), 'Sorten,', len(gerichte), 'Gerichte,', aufrufe[0], 'Aufrufe')
    print('Höchstens', len(gekocht), 'Gerichte sind kochbar, zum Beispiel mit diesem Plan:')
    zeige_plan(plan)
    for g in gekocht:
        monate = []
        for monat in range(12):
            if gericht_im_monat(plan, monat, gerichte) == g:
                monate.append(MONATE[monat])
        print('  ' + ', '.join(g) + ':', ', '.join(monate))
    print()
    return plan, gekocht


ergebnisse = {}
for nr in range(1, 7):
    ergebnisse[nr] = bearbeite('beispieldaten/bauern' + str(nr) + '.txt')

beispieldaten/bauern1.txt: 4 Sorten, 4 Gerichte, 27 Aufrufe
Höchstens 4 Gerichte sind kochbar, zum Beispiel mit diesem Plan:
Monat Anlage 1   Anlage 2   Anlage 3
Jan   Möhre      Zwiebel    Kartoffel  
Feb   Möhre      Zwiebel    Reis       
Mär   Möhre      Kartoffel  Reis       
Apr   Zwiebel    Kartoffel  Reis       
Mai   Zwiebel    Kartoffel  beliebig   
Jun   Zwiebel    beliebig   beliebig   
Jul   beliebig   beliebig   beliebig   
Aug   beliebig   beliebig   beliebig   
Sep   beliebig   beliebig   beliebig   
Okt   beliebig   beliebig   beliebig   
Nov   beliebig   beliebig   Kartoffel  
Dez   beliebig   Zwiebel    Kartoffel  
  Kartoffel, Zwiebel, Möhre: Jan
  Zwiebel, Möhre, Reis: Feb
  Kartoffel, Möhre, Reis: Mär
  Kartoffel, Zwiebel, Reis: Apr

beispieldaten/bauern2.txt: 8 Sorten, 6 Gerichte, 362 Aufrufe
Höchstens 6 Gerichte sind kochbar, zum Beispiel mit diesem Plan:
Monat Anlage 1     Anlage 2     Anlage 3
Jan   Möhre        Zwiebel      Kartoffel    
Feb   Möhre        Zw

**Ergebnis:** Die Anzahlen stimmen mit der Tabelle in den offiziellen Lösungshinweisen überein:

| Beispiel | Gerichte | höchstens kochbar |
|---|---|---|
| bauern1 | 4 | 4 |
| bauern2 | 6 | 6 |
| bauern3 | 19 | 8 |
| bauern4 | 36 | 9 |
| bauern5 | 50 | 9 |
| bauern6 | 100 | 10 |

Die Pläne können anders aussehen als in den Lösungshinweisen: Es gibt fast immer mehrere beste Pläne, schon wegen der
Verschiebung. „beliebig“ heißt, dass die Pflanze in diesem Slot für die gewählten Gerichte keine Rolle spielt. Man darf dort
irgendeine Sorte anbauen.

## Für Neugierige

**Mehr als 12 geht nie:** Jeder Monat liefert höchstens ein Gericht. Bei `bauern6.txt` mit 100 Gerichten sind höchstens 10
gleichzeitig möglich.

**Die Lösungshinweise** gehen genauso vor (Gerichte statt Pflanzen, Schranke, Verschiebung), aber mit der einfachen Schranke aus
Schritt 5. Unsere Spiegelung und die bessere Schranke mit `noch_moeglich` stehen dort nicht.

**Weitere Ideen:**
- **Gute Reihenfolge:** Gerichte, die viele Zutaten mit anderen Gerichten teilen, zuerst probieren. Dann findet man früher einen
  guten Plan, und die Schranke greift eher.
- **Noch bessere Schranke:** In einem Monat kann nur ein Gericht dazukommen, und jedes Gericht zählt nur einmal. Wie viele Monate
  man höchstens mit **verschiedenen** offenen Gerichten füllen kann, ist ein Zuordnungsproblem (*bipartites Matching*). Das ergibt
  eine noch genauere Obergrenze, ist aber aufwendiger zu programmieren.
- **Ganzzahlige Optimierung und Constraint-Solver:** Die Aufgabe lässt sich auch als Modell für einen Solver beschreiben. Wie das
  mit MiniZinc geht, zeigt der letzte Abschnitt.

## Zum Schluss: eine Lösung mit MiniZinc

Der Plan als Kreis aus 12 Pflanzen passt gut zu MiniZinc: 12 Entscheidungsvariablen, und der Solver soll möglichst viele Gerichte
kochbar machen. Die MiniZinc-Blöcke unten sind **kein Python**: Kopiere sie in die MiniZinc IDE (oder in den
[MiniZinc Playground](https://play.minizinc.org/)) und führe sie dort aus.

#### Die Daten

MiniZinc bekommt die Sorten als Liste von Namen und die Gerichte als Tabelle der Nummern ihrer Zutaten (ab 1 gezählt). Die
folgende Zelle schreibt den Inhalt der Datendatei.

In [ ]:
def als_dzn(dateiname):
    gemuese, gerichte = lies_eingabe(dateiname)
    zeilen = []
    for g in gerichte:
        nummern = []
        for zutat in g:
            nummern.append(gemuese.index(zutat) + 1)
        zeilen.append(str(nummern)[1:-1])          # ohne die eckigen Klammern
    text = 'n = ' + str(len(gemuese)) + ';\n'
    text = text + 'name = ' + str(gemuese).replace("'", '"') + ';\n'
    text = text + 'm = ' + str(len(gerichte)) + ';\n'
    text = text + 'gericht = [|' + '\n          |'.join(zeilen) + '|];'
    return text


print(als_dzn('beispieldaten/bauern1.txt'))

n = 4;
name = ["Kartoffel", "Zwiebel", "Möhre", "Reis"];
m = 4;
gericht = [|1, 2, 3
          |1, 2, 4
          |1, 3, 4
          |2, 3, 4|];


#### Das Modell

Wie die Python-Suche wählt das Modell für jeden Monat ein Gericht `neu[mon]` oder keins (0).

- `alldifferent_except_0(neu)` ist wie `alldifferent`, nur dass die 0 beliebig oft vorkommen darf. So wird jedes Gericht
  höchstens einmal gezählt.
- Wurde im Monat `mon` ein Gericht gewählt, muss jede seiner drei Zutaten in einem der Slots `mon`, `mon − 1` oder `mon − 2`
  wachsen. `+ 12` vor dem `mod 12` sorgt dafür, dass im Januar nicht mit negativen Zahlen gerechnet wird. `gericht[neu[mon], i]`
  benutzt eine Variable als Index (Arbeitsblatt *Grundlagen*, Abschnitt 7).
- Wie in Schritt 5 brechen wir die Verschiebungssymmetrie: Das Gericht mit der größten Nummer soll im Januar gekocht werden. Durch
  Verschieben kann man das bei jedem Plan erreichen. Bei den meisten Beispielen wird der Solver dadurch deutlich schneller
  (ohne diese Zeile wird er bei `bauern6.txt` in zwei Minuten nicht fertig), nur bei `bauern5.txt` ist er ohne sie schneller.
- Die Ausgabe ist etwas länger. `slot_von` und `spalte` sind kleine **Funktionen** in MiniZinc (wie `def` in Python):
  `slot_von` rechnet wie in Schritt 2, `spalte` hängt so viele Leerzeichen an einen Namen, dass er `breite` Zeichen lang ist
  (wie `ljust` in Python). `concat` hängt eine Liste von Texten aneinander, `join(", ", liste)` dasselbe mit Komma dazwischen.

```minizinc
include "globals.mzn";

% ----- Daten (aus der .dzn-Datei) -----
int: n;                                   % Anzahl der Sorten
array[1..n] of string: name;
int: m;                                   % Anzahl der Gerichte
array[1..m, 1..3] of 1..n: gericht;       % gericht[g, i]: Nummer der i-ten Zutat von Gericht g

set of int: MONAT = 0..11;                % 0 = Januar; zugleich die Nummern der Slots

% ----- Entscheidungsvariablen -----
array[MONAT] of var 1..n: plan;           % plan[s]: Sorte im Slot s (Zyklus, der im Monat s beginnt)
array[MONAT] of var 0..m: neu;            % neu[mon]: Gericht, das im Monat mon gekocht wird (0: keins)

% ----- Bedingungen -----
% kein Gericht wird doppelt gezählt (0 darf mehrfach vorkommen)
constraint alldifferent_except_0(neu);

% jede Zutat des gewählten Gerichts wächst in einem der drei Slots mon, mon - 1, mon - 2
% (bei neu[mon] = 0 wird nichts verlangt, weil die linke Seite von -> falsch ist)
constraint forall(mon in MONAT, i in 1..3)(
    neu[mon] > 0 -> exists(j in 0..2)(plan[(mon - j + 12) mod 12] = gericht[neu[mon], i])
);

% Verschiebung: Das Gericht mit der größten Nummer wird im Januar gekocht
constraint forall(mon in 1..11)(neu[mon] <= neu[0]);

var int: anzahl = sum(mon in MONAT)(neu[mon] > 0);
solve maximize anzahl;

% ----- Ausgabe: Monate untereinander, dazu die Anlagen und das neue Gericht -----
array[1..12] of string: monatsname = ["Jan", "Feb", "Mär", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"];
array[1..3] of int: erster_slot = [0, 2, 1];
int: breite = max(s in 1..n)(string_length(name[s])) + 2;
function int: slot_von(int: anlage, int: mon) = (mon - (mon - erster_slot[anlage] + 3) mod 3 + 12) mod 12;
function string: spalte(string: text) = text ++ concat([" " | k in 1..breite - string_length(text)]);

output ["Kochbare Gerichte: \(anzahl)\n"] ++
       [monatsname[mon + 1] ++ "  " ++ concat([spalte(name[fix(plan[slot_von(a, mon)])]) | a in 1..3]) ++
        if fix(neu[mon]) > 0 then join(", ", [name[gericht[fix(neu[mon]), i]] | i in 1..3]) else "" endif ++ "\n"
        | mon in MONAT];
```

**Wähle als Solver Chuffed.** Gecode findet bei den größeren Beispielen auch nach einer Minute nicht den besten Plan. Mit Chuffed
und den Daten von `bauern3.txt`:

**Ausgabe:**

```
Kochbare Gerichte: 8
Jan  Zucchini       Linse          Pastinake      Linse, Zucchini, Pastinake
Feb  Zucchini       Linse          Paprika        Linse, Zucchini, Paprika
Mär  Zucchini       Kartoffel      Paprika        Kartoffel, Zucchini, Paprika
Apr  Zwiebel        Kartoffel      Paprika        Kartoffel, Zwiebel, Paprika
Mai  Zwiebel        Kartoffel      Möhre          Kartoffel, Zwiebel, Möhre
Jun  Zwiebel        Paprika        Möhre          
Jul  Tomate         Paprika        Möhre          Möhre, Paprika, Tomate
Aug  Tomate         Paprika        Zwiebel        
Sep  Tomate         Rosenkohl      Zwiebel        Rosenkohl, Tomate, Zwiebel
Okt  Kartoffel      Rosenkohl      Zwiebel        
Nov  Kartoffel      Rosenkohl      Pastinake      Kartoffel, Pastinake, Rosenkohl
Dez  Kartoffel      Linse          Pastinake      
----------
==========
```

In den freien Slots steht hier immer irgendeine Sorte (MiniZinc muss jeder Variablen einen Wert geben).

Für alle Beispiele (erste Zeile der Ausgabe):

| Beispiel | Kochbare Gerichte |
|---|---|
| bauern1 | 4 |
| bauern2 | 6 |
| bauern3 | 8 |
| bauern4 | 9 |
| bauern5 | 9 |
| bauern6 | 10 |

Das sind dieselben Zahlen wie in Schritt 7. Für `bauern1.txt` bis `bauern4.txt` braucht Chuffed nur wenige Sekunden, für die
beiden Herausforderungen `bauern5.txt` und `bauern6.txt` eine halbe bis eine ganze Minute, also länger als unser Python-Programm.

Auch hier steckt Wissen über die Aufgabe im Modell. Ein Modell, das für jedes Gericht nur fragt, ob es in irgendeinem Monat
kochbar ist, ist kürzer. Damit findet Chuffed aber bei `bauern3.txt` auch nach einer Minute nicht den besten Plan. Erst die Wahl
eines Gerichts pro Monat (wie bei der Python-Suche) und die Verschiebungssymmetrie machen den Solver schnell genug.